# Step 3 — Merge workflow records with cleaned repository metadata

> Join workflows to their parent-repository metadata (stars, language, dates, commit count) so subsequent filters can reason about both levels in one frame.

**Paper section:** Step 1 — Repository selection and workflow extraction (offline preparation).

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/a-dataset-of-github-actions-workflow-histories-workflows.csv` | CSV | Raw workflow histories (external dataset) |
| `data/1_cleaned_repositories.csv` | CSV | Step 1 |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/3_merged_dataset.csv` | CSV (workflows joined with repo metadata) | Step 4 (final cleaning) |
| `data/step3_report.json` | JSON | Step 4 (integrity check) |



In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 3 — starting: merge workflows with repositories...", flush=True)
"""Step 3: prepare workflow snapshot and merge with cleaned repositories.

Reads the raw workflows CSV (3.4M history rows), extracts the most recent
valid entry per uid (filtering deleted and invalid workflows), then performs
an inner join with the cleaned repositories on repository name.

This step absorbs the former deduplication step — extracting the current
state of each workflow is a data preparation concern, not a separate
research step. The repository-level and workflow-level criteria applied
downstream are what yield unique, curated repositories and workflows.

Outputs:
    data/3_merged_dataset.csv          Merged dataset
  data/step3_report.json           Statistics for paper reporting
"""

import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import polars as pl


@dataclass
class MergeStats:
    # Workflow snapshot
    raw_workflow_rows: int = 0
    removed_invalid_workflow: int = 0
    removed_deleted: int = 0
    removed_history_duplicates: int = 0
    removed_path_duplicates: int = 0
    unique_workflows: int = 0
    # Repository
    total_repos: int = 0
    # Merge
    matched: int = 0
    unmatched_workflows: int = 0
    unmatched_repos: int = 0

def _extract_workflow_snapshot(df: pl.DataFrame, stats: MergeStats) -> pl.DataFrame:
    """Extract most recent valid entry per uid from workflow history."""
    stats.raw_workflow_rows = len(df)

    # valid_workflow may be Boolean or String depending on Polars inference
    vw_col = df["valid_workflow"]
    if vw_col.dtype == pl.Boolean:
        invalid_mask = ~pl.col("valid_workflow")
    else:
        invalid_mask = pl.col("valid_workflow") != True
    stats.removed_invalid_workflow = df.filter(invalid_mask).height
    df = df.filter(~invalid_mask)

    stats.removed_deleted = df.filter(pl.col("git_change_type") == "D").height
    df = df.filter(pl.col("git_change_type") != "D")

    before_dedup = len(df)
    # Deterministic ordering: committed_date DESC then uid/file_path ASC to
    # break ties consistently across runs (polars sort is not stable by default).
    df = df.sort(
        ["committed_date", "uid", "file_path"],
        descending=[True, False, False],
    )
    df = df.unique(subset=["uid"], keep="first", maintain_order=True)
    stats.removed_history_duplicates = before_dedup - len(df)

    # A file renamed multiple times can produce different uid values that all
    # resolve to the same (repository, file_path) destination.  Keep only the
    # most-recent commit per path to guarantee one row per workflow file.
    before_path_dedup = len(df)
    df = df.unique(
        subset=["repository", "file_path"], keep="first", maintain_order=True
    )
    stats.removed_path_duplicates = before_path_dedup - len(df)

    stats.unique_workflows = len(df)

    return df

def merge_datasets(
    workflows_path: Path,
    repositories_path: Path,
    output_path: Path,
    report_path: Path,
) -> MergeStats:
    stats = MergeStats()

    from dataset_utils.integrity import assert_input_matches_report, hash_source_file

    input_workflows_sha = hash_source_file(workflows_path, "step 3 workflows source")
    input_repos_sha = assert_input_matches_report(
        input_path=repositories_path,
        prior_report_path=Path("data/step1_report.json"),
        label="step 3 repositories input",
    )

    # Load and snapshot workflows
    print("  reading input CSV (may take a moment for large files)...", flush=True)
    workflows_df = pl.read_csv(workflows_path, encoding="utf-8-sig")
    workflows_df = _extract_workflow_snapshot(workflows_df, stats)

    # Load repositories
    print("  reading input CSV (may take a moment for large files)...", flush=True)
    repos_df = pl.read_csv(repositories_path, encoding="utf-8-sig")
    stats.total_repos = len(repos_df)

    # Prepare join keys
    workflows_df = workflows_df.rename({"repository": "repo_name"})
    repos_for_join = repos_df.rename({"name": "repo_name"})
    for col in repos_for_join.columns:
        if col != "repo_name":
            repos_for_join = repos_for_join.rename({col: f"repo_{col}"})

    # Inner join
    merged_df = workflows_df.join(repos_for_join, on="repo_name", how="inner")
    stats.matched = len(merged_df)
    stats.unmatched_workflows = stats.unique_workflows - stats.matched

    all_repo_names = set(repos_df["name"].to_list())
    matched_repo_names = set(merged_df["repo_name"].to_list())
    stats.unmatched_repos = len(all_repo_names - matched_repo_names)

    # Restore original column name for downstream steps
    merged_df = merged_df.rename({"repo_name": "repository"})

    # Write outputs
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("  writing output...", flush=True)
    merged_df.write_csv(output_path, include_header=True)

    from dataset_utils.hashing import compute_file_hash
    report = {
        **asdict(stats),
        "input_workflows_sha256": input_workflows_sha,
        "input_repositories_sha256": input_repos_sha,
        "output_file": str(output_path),
        "output_sha256": compute_file_hash(output_path),
    }
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open("w") as f:
        json.dump(report, f, indent=2)

    return stats

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Extract workflow snapshot and merge with cleaned repositories."
    )
    parser.add_argument(
        "--workflows",
        type=Path,
        default=Path("data/a-dataset-of-github-actions-workflow-histories-workflows.csv"),
        help="Input raw workflows CSV file path.",
    )
    parser.add_argument(
        "--repositories",
        type=Path,
        default=Path("data/1_cleaned_repositories.csv"),
        help="Input cleaned repositories CSV file path.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/3_merged_dataset.csv"),
        help="Output merged CSV file path.",
    )
    parser.add_argument(
        "--report",
        type=Path,
        default=Path("data/step3_report.json"),
        help="Output report JSON file path.",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    stats = merge_datasets(args.workflows, args.repositories, args.output, args.report)

    print("Workflow snapshot + merge finished.")
    print(f"Workflows input: {args.workflows}")
    print(f"Repositories input: {args.repositories}")
    print(f"Output: {args.output}")
    print(f"Report: {args.report}")
    print()
    print("Workflow snapshot:")
    print(f"  Raw history rows:       {stats.raw_workflow_rows:,}")
    print(f"  Removed (invalid):      {stats.removed_invalid_workflow:,}")
    print(f"  Removed (deleted):      {stats.removed_deleted:,}")
    print(f"  Removed (history dups): {stats.removed_history_duplicates:,}")
    print(f"  Removed (path dups):    {stats.removed_path_duplicates:,}")
    print(f"  Unique workflows:       {stats.unique_workflows:,}")
    print()
    print("Merge:")
    print(f"  Repositories:           {stats.total_repos:,}")
    print(f"  Matched rows:           {stats.matched:,}")
    print(f"  Unmatched workflows:    {stats.unmatched_workflows:,}")
    print(f"  Unmatched repos:        {stats.unmatched_repos:,}")

main()


Step 3 — starting: merge workflows with repositories...
  reading input CSV (may take a moment for large files)...
  reading input CSV (may take a moment for large files)...
  writing output...
Workflow snapshot + merge finished.
Workflows input: data/a-dataset-of-github-actions-workflow-histories-workflows.csv
Repositories input: data/1_cleaned_repositories.csv
Output: data/3_merged_dataset.csv
Report: data/step3_report.json

Workflow snapshot:
  Raw history rows:       3,418,911
  Removed (invalid):      181,367
  Removed (deleted):      70,358
  Removed (history dups): 2,906,742
  Removed (path dups):    6,519
  Unique workflows:       253,925

Merge:
  Repositories:           49,510
  Matched rows:           253,925
  Unmatched workflows:    0
  Unmatched repos:        528


---

**Next step:** [`04_final_cleaning.ipynb`](./04_final_cleaning.ipynb)
